In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIGURATION PK/PD (Modèle de Schnider) ---
V1, V2, V3 = 4.27, 18.9, 238.0
k10, k12, k21, k13, k31 = 0.38, 0.30, 0.20, 0.19, 0.0035
ke0 = 0.17  # Constante de transfert vers le site d'effet [8]

# Paramètres PD (Equation de Hill pour le BIS)
BIS_0, BIS_MAX, EC50, HILL = 95.0, 75.0, 3.5, 2.5 # [9]
BIS_TARGET = 50.0

# --- CONFIGURATION RL ---
ACTIONS = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0] # Version réduite pour le calcul [10]
GAMMA = 0.69 # Facteur de remise [11]
BINS_PER_FEAT = 10
NUM_STATES = BINS_PER_FEAT**3 # 3 fonctions floues (N, Z, P)

# --- ARTIFACT PERSISTENCE ---
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DP_PATH = ARTIFACTS_DIR / "dp_deltabis_agent.npz"

def get_fuzzy_features(deltabis):
    """Fuzzification de deltaBIS selon Moore et al. (Fig 1)"""
    x = 20.0 # Échelle de l'erreur
    v = np.clip(deltabis / x, -1, 1)
    mu_n = max(0, -v)
    mu_p = max(0, v)
    mu_z = max(0, 1 - abs(v))
    return np.array([mu_n, mu_z, mu_p])

def state_to_idx(features):
    """Map the 3D fuzzy feature vector to an integer index in 0..NUM_STATES-1.

    Encode bins in base `BINS_PER_FEAT`: index = b0*BINS_PER_FEAT^2 + b1*BINS_PER_FEAT + b2
    """
    features = np.asarray(features)
    if features.shape != (3,):
        raise ValueError(f"features must be length-3 vector, got shape {features.shape}")
    # discretize to 0..BINS_PER_FEAT-1 safely
    bins = np.clip((features * (BINS_PER_FEAT - 1)), 0, BINS_PER_FEAT - 1).astype(int)
    # encode in base BINS_PER_FEAT
    idx = int(bins[0] * (BINS_PER_FEAT**2) + bins[1] * BINS_PER_FEAT + bins[2])
    return idx

def get_ce_from_error(error):
    """Inverse de la fonction PD pour estimer Ce à partir du BIS"""
    bis = error + BIS_TARGET
    # Inversion de BIS = BIS_0 - BIS_MAX * (Ce^H / (Ce^H + EC50^H))
    ratio = (BIS_0 - bis) / BIS_MAX
    ratio = np.clip(ratio, 0.01, 0.99)
    ce = EC50 * (ratio / (1 - ratio))**(1/HILL)
    return ce

# --- DP ALGORITHM ---
# Always compute/update agent, but can warm-start from existing file
loaded_V = None
if DP_PATH.exists():
    try:
        data = np.load(DP_PATH)
        loaded_V = data["V"]
        print(f"Found existing agent at {DP_PATH}, will warm-start Value Iteration.")
    except Exception as e:
        print(f"Could not load existing agent: {e}")

print("Computing Transition Matrix (P) and Rewards (R)...")
P = np.zeros((NUM_STATES, len(ACTIONS)), dtype=int)
R = np.zeros((NUM_STATES, len(ACTIONS)))

# Iterate directly over states and actions (much faster: 1000 × 7 = 7000 iterations)
for s in range(NUM_STATES):
    # Decode state index back to feature bins
    bins = np.array([s // (BINS_PER_FEAT**2),
                     (s // BINS_PER_FEAT) % BINS_PER_FEAT,
                     s % BINS_PER_FEAT], dtype=float)
    features = bins / (BINS_PER_FEAT - 1)  # Normalize back to [0, 1]

    # Approximate deltaBIS from fuzzy features (inverse of get_fuzzy_features)
    # Simplified: use the negative feature as a proxy for deltaBIS range
    deltabis = (features[0] - features[2]) * 30  # Range approximately -30 to 30

    for a_idx, infusion in enumerate(ACTIONS):
        # Simulate next deltaBIS based on current infusion and deltaBIS
        # Higher infusion -> BIS decreases -> next deltaBIS becomes more negative
        deltabis_next = deltabis - (infusion / max(ACTIONS)) * 10
        deltabis_next = np.clip(deltabis_next, -30, 30)

        s_next = state_to_idx(get_fuzzy_features(deltabis_next))

        # Reward based on how far deltaBIS is from target (0 is target)
        R[s, a_idx] = -abs(deltabis_next)
        P[s, a_idx] = s_next

# 2. ALGORITHME DE PROGRAMMATION DYNAMIQUE (Value Iteration)
print("Exécution de Value Iteration...")
if loaded_V is not None and loaded_V.shape == (NUM_STATES,):
    V = loaded_V.copy()
else:
    V = np.zeros(NUM_STATES)

for i in range(10000):
    V_old = V.copy()
    # Vectorized computation: Q(s,a) = R(s,a) + gamma * V(P(s,a))
    Q_vals = R + GAMMA * V[P]  # Shape: (NUM_STATES, len(ACTIONS))
    V = np.max(Q_vals, axis=1)  # Max over actions

    if np.max(np.abs(V - V_old)) < 1e-4:
        print(f"Converged at iteration {i}")
        break

    if (i + 1) % 100 == 0:
        print(f"  Iteration {i+1}, max change: {np.max(np.abs(V - V_old)):.6f}")

# 3. EXTRACTION DE LA POLITIQUE OPTIMALE
print("Extracting optimal policy...")
# Vectorized: Q(s,a) = R(s,a) + gamma * V(P(s,a))
Q_vals = R + GAMMA * V[P]  # Shape: (NUM_STATES, len(ACTIONS))
policy = np.argmax(Q_vals, axis=1).astype(int)  # Best action per state

# --- SAVE DP AGENT ---
np.savez_compressed(
    DP_PATH,
    V=V,
    policy=policy,
    P=P,
    R=R,
    actions=np.array(ACTIONS, dtype=float),
    gamma=np.array([GAMMA], dtype=float),
)
print(f"Saved DP agent to {DP_PATH}")

# --- VISUALISATION ---
# Configuration for evaluation episodes (clear, adjustable)
NUM_EVAL_EPISODES = 10  # number of simulated evaluation episodes (reduced for speed - clear variable)
EP_LEN = 720            # episode length in time steps (12 minutes = 720 steps of 5s each)
DT = 5 / 60             # time step in minutes (5s)
np.random.seed(0)

# Plot policy as function of deltaBIS (existing small-figure plot)
deltabis_range = np.linspace(-30, 30, 100)
chosen_doses = []
for db in deltabis_range:
    s = state_to_idx(get_fuzzy_features(db))
    chosen_doses.append(ACTIONS[policy[s]])

plt.figure(figsize=(10, 3))
plt.plot(deltabis_range, chosen_doses, color='blue', lw=2)
plt.title("Politique Optimale DP (Dose de Propofol vs deltaBIS)")
plt.xlabel("deltaBIS (BIS(t+1) - BIS(t-1))")
plt.ylabel("Infusion Rate (ml/min)")
plt.grid(True, linestyle='--')
plt.axvline(0, color='red', linestyle='-', alpha=0.3, label="Target")
plt.legend()
plt.show()

# Evaluate policy with several simulated episodes and track BIS and actions
print(f"Evaluating policy for {NUM_EVAL_EPISODES} episodes, each {EP_LEN} steps ({DT*60:.0f}s per step)...")
all_bis = np.zeros((NUM_EVAL_EPISODES, EP_LEN))
all_actions = np.zeros((NUM_EVAL_EPISODES, EP_LEN))
c0,c1,c2 = 0.0, 0.0, 0.0  # Initial concentrations in compartments

for ep in range(NUM_EVAL_EPISODES):
    # initialize episode: start at target BIS or small perturbation
    bis_prev = BIS_TARGET if ep > 0 else (BIS_TARGET - 5.0)  # give first episode a slight offset for variety
    bis_curr = bis_prev
    deltabis = 0.0  # Initialize deltaBIS to 0
    ce = get_ce_from_error(bis_curr - BIS_TARGET)    # estimate effect-site conc from error

    for t in range(EP_LEN):
        s = state_to_idx(get_fuzzy_features(deltabis))
        a_idx = int(policy[s])
        infusion = ACTIONS[a_idx]

        # 3-compartment PK Euler update (per-minute rates, DT in minutes)
        dc0 = (infusion / V1) - (k10 + k12 + k13) * c0 + k21 * c1 + k31 * c2
        dc1 = k12 * c0- k21 * c1
        dc2 = k13 * c0 - k31 * c2

        # integrate
        c0 += dc0 * DT
        c1 += dc1 * DT
        c2 += dc2 * DT

        # effect-site equilibration toward central compartment
        ce += ke0 * (c0 - ce) * DT

        # pharmacodynamic BIS from effect-site
        bis = BIS_0 - BIS_MAX * (ce**HILL / (ce**HILL + EC50**HILL))

        # Compute deltaBIS = BIS(t+1) - BIS(t-1)
        bis_prev_val = bis_prev
        bis_prev = bis_curr
        bis_curr = bis
        deltabis = bis_curr - bis_prev_val

        all_bis[ep, t] = bis
        all_actions[ep, t] = infusion

# Plot BIS trajectories: mean ± std and sample episodes
mean_bis = all_bis.mean(axis=0)
std_bis = all_bis.std(axis=0)

t = np.arange(EP_LEN) * (DT * 60)  # time in seconds
plt.figure(figsize=(10, 5))
plt.plot(t, mean_bis, color='blue', label='Mean BIS')
plt.fill_between(t, mean_bis - std_bis, mean_bis + std_bis, color='blue', alpha=0.2, label='±1 std')
# overlay a couple of sample episodes
for ep in range(min(3, NUM_EVAL_EPISODES)):
    plt.plot(t, all_bis[ep], alpha=0.6, lw=1, label=f'Episode {ep+1}')

plt.axhline(BIS_TARGET, color='red', linestyle='--', label='Target BIS')
plt.xlabel('Time (s)')
plt.ylabel('BIS')
plt.title('Policy Evaluation: BIS trajectories (mean ± std and samples)')
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

# Plot corresponding infusion traces (first few episodes)
plt.figure(figsize=(10, 4))
for ep in range(min(4, NUM_EVAL_EPISODES)):
    plt.step(t, all_actions[ep], where='post', alpha=0.8, label=f'Ep {ep+1}')
plt.xlabel('Time (s)')
plt.ylabel('Infusion (ml/min)')
plt.title('Actions taken by policy during evaluation episodes')
plt.legend()
plt.grid(True, linestyle='--')
plt.show()


# EVALUATION ON POPULATION DATASET

In [ ]:
import pandas as pd
import random

# 1. Data Loading and Preprocessing
def load_data(path: str):
    return pd.read_csv(path)

def preprocess_data(df: pd.DataFrame):
    df = df.copy()
    def parse_age(x):
        try:
            parts = str(x).strip().split(" ")
            if len(parts) < 2: return 50
            low = int(parts[1])
            if len(parts) > 3:
                if parts[3] == 'older':
                    high = 100
                else:
                    high = int(parts[3])
                return int(random.randrange(low, high))
            return int(low)
        except Exception:
            return 50 # Default fallback

    if "AgeCategory" in df.columns:
        df.loc[:,"AgeCategory"] = df["AgeCategory"].apply(parse_age)
    return df

def schnider_model(age: int, weight: float, height: float, sex: str) -> dict:
    sex = str(sex).lower()
    if sex == "male":
        lbm = 1.10 * weight - 128 * (weight ** 2) / (height ** 2)
    else:
        lbm = 1.07 * weight - 148 * (weight ** 2) / (height ** 2)

    V1 = 4.27
    V2 = 18.9 - 0.391 * (age - 53)
    V3 = 238.0

    k10 = 0.443 + 0.0107 * (weight - 77) - 0.0159 * (lbm - 59) + 0.0062 * (height - 177)
    k12 = 0.302 - 0.0056 * (age - 53)
    k13 = 0.196
    k21 = (1.29 - 0.024 * (age - 53)) / V2
    k31 = 0.0035
    ke0 = 0.456

    A = np.array([
        [-(k10 + k12 + k13),  k21,  k31,   0   ],
        [        k12,         -k21,   0,    0   ],
        [        k13,          0,   -k31,   0   ],
        [        ke0,          0,    0,   -ke0  ]
    ], dtype=float)
    B = np.array([[1 / V1], [0], [0], [0]])
    return {"A": A, "B": B}

def generate_schnider_dataset(df: pd.DataFrame):
    # Fusing parameters to each row safely
    params_list = []
    for _, row in df.iterrows():
        params = schnider_model(
            age=row["AgeCategory"],
            weight=row["WeightInKilograms"],
            height=row["HeightInMeters"],
            sex=row["Sex"]
        )
        params_list.append(params)

    # Create DataFrame from list of dicts
    params_df = pd.DataFrame(params_list)
    # Concatenate with original dataframe
    return pd.concat([df.reset_index(drop=True), params_df], axis=1)

# 2. Evaluator specific to DP Agent (1000 states, 3 features)
class DPEvaluator:
    def __init__(self, policy, actions):
        self.policy = policy
        self.actions = actions
        self.target = 50.0
        self.bis0 = 95.0
        self.bis_max = 75.0
        self.ec50 = 3.5
        self.hill = 2.5
        # Match BINS_PER_FEAT from DP part
        self.bins_per_feat = 10

    def _get_state_idx(self, deltabis):
        """Match feature extraction from DP training part exactly"""
        x = 20.0
        v = np.clip(deltabis / x, -1, 1)
        mu_n = max(0, -v)
        mu_p = max(0, v)
        mu_z = max(0, 1 - abs(v))
        features = np.array([mu_n, mu_z, mu_p])

        # Use same encoding as state_to_idx in DP training
        bins = np.clip((features * (self.bins_per_feat - 1)), 0, self.bins_per_feat - 1).astype(int)
        idx = int(bins[0] * (self.bins_per_feat**2) + bins[1] * self.bins_per_feat + bins[2])
        # Clamp to policy size
        idx = min(idx, len(self.policy) - 1)
        return idx

    def simulate(self, patient_row, duration_min=120):
        dt = 5/60
        steps = int(duration_min / dt)

        # Extract A and B matrices safely (convert from object arrays if needed)
        A = np.asarray(patient_row['A'], dtype=float)
        B = np.asarray(patient_row['B'], dtype=float)

        # Initialize state vector realistically: start at baseline (BIS=95, ce=0)
        x = np.zeros((4, 1), dtype=float)

        pe_log = []
        bis_log = []
        bis_prev_prev = self.bis0
        bis_prev = self.bis0

        for t in range(steps):
            # BIS Output from effect-site concentration
            ce = max(0.0, x[3, 0])  # Effect-site conc, clamp to >= 0

            # Hill equation: BIS = BIS_0 - BIS_MAX * (Ce^H / (Ce^H + EC50^H))
            ce_h = ce ** self.hill
            ec50_h = self.ec50 ** self.hill
            bis_ideal = self.bis0 - self.bis_max * (ce_h / (ce_h + ec50_h))

            # Handle NaN from numerical issues
            if np.isnan(bis_ideal):
                bis_ideal = self.bis0

            # Clamp BIS to valid range and add measurement noise
            bis_ideal = np.clip(bis_ideal, 0, 100)
            measured_bis = bis_ideal + np.random.normal(0, 3)
            measured_bis = np.clip(measured_bis, 0, 100)

            # RL Perception - compute deltaBIS = BIS(t+1) - BIS(t-1)
            deltabis = measured_bis - bis_prev_prev if t > 0 else 0.0
            s_idx = self._get_state_idx(deltabis)

            # Action selection
            a_idx = int(self.policy[s_idx])
            u = float(self.actions[a_idx])

            # PK update: dx/dt = Ax + Bu
            x_dot = A @ x + B * u
            x = x + x_dot * dt

            # Track for deltaBIS computation
            bis_prev_prev = bis_prev
            bis_prev = measured_bis

            # Compute error for metrics
            error = measured_bis - self.target
            pe_log.append(100.0 * error / self.target if self.target != 0 else 0.0)
            bis_log.append(float(measured_bis))

        return bis_log, pe_log

    def evaluate(self, df):
        summary = []
        for idx, row in df.iterrows():
            try:
                bis, pe = self.simulate(row)
                mdpe = np.median(pe)
                mdape = np.median(np.abs(pe))
                wobble = np.median(np.abs(pe - mdpe))
                controlled = (np.abs(np.array(bis) - self.target) <= 5).mean() * 100
                summary.append({
                    'PatientID': row['PatientID'],
                    'MDPE': mdpe, 'MDAPE': mdape,
                    'Wobble': wobble, 'Controlled (%)': controlled
                })
            except Exception as e:
                print(f"Warning: Failed to evaluate patient {row['PatientID']}: {e}")
                continue
        return pd.DataFrame(summary)

# 3. Execution
EVAL_SAMPLE_SIZE = 50  # Adjust as needed

try:
    print("Loading Population Data...")
    raw_data = load_data("./data/Patients Data.csv")
    cols = ["PatientID", "Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"]
    df_clean = preprocess_data(raw_data[cols])

    if EVAL_SAMPLE_SIZE and len(df_clean) > EVAL_SAMPLE_SIZE:
        print(f"Sampling {EVAL_SAMPLE_SIZE} patients from {len(df_clean)} total.")
        df_clean = df_clean.sample(n=EVAL_SAMPLE_SIZE, random_state=42)

    df_sim = generate_schnider_dataset(df_clean)

    print("Loading DP Agent...")
    dp_data = np.load(DP_PATH)
    loaded_policy = dp_data["policy"]
    loaded_actions = dp_data["actions"]

    print("Evaluating on Population...")
    evaluator = DPEvaluator(loaded_policy, loaded_actions)
    results_df = evaluator.evaluate(df_sim)

    print("\n--- Evaluation Results Summary ---")
    print(results_df[['MDPE', 'MDAPE', 'Wobble', 'Controlled (%)']].describe())

    # Optional: Plot best/worst
    best_pt = results_df.loc[results_df['MDAPE'].idxmin()]
    print(f"\nBest Patient: {best_pt.to_dict()}")

except Exception as e:
    print(f"Evaluation failed: {e}")


# LOAD AND EVALUATE SAVED DP AGENT

In [ ]:
print("=" * 60)
print("LOADING SAVED DP AGENT")
print("=" * 60)

# Check if agent file exists
if not DP_PATH.exists():
    print(f"ERROR: Agent file not found at {DP_PATH}")
    print("Please run the DP training cell first.")
else:
    try:
        # Load saved agent
        print(f"\nLoading agent from: {DP_PATH}")
        agent_data = np.load(DP_PATH)

        print(f"  - Loaded V (value function): shape {agent_data['V'].shape}")
        print(f"  - Loaded policy: shape {agent_data['policy'].shape}")
        print(f"  - Loaded P (transitions): shape {agent_data['P'].shape}")
        print(f"  - Loaded R (rewards): shape {agent_data['R'].shape}")
        print(f"  - Actions available: {agent_data['actions']}")
        print(f"  - Gamma (discount): {agent_data['gamma']}")

        saved_policy = agent_data["policy"]
        saved_actions = agent_data["actions"]

        # Create evaluator with loaded policy
        evaluator = DPEvaluator(saved_policy, saved_actions)

        # Load and prepare population data
        print("\n" + "=" * 60)
        print("EVALUATING ON POPULATION SAMPLE")
        print("=" * 60)

        raw_data = load_data("./data/Patients Data.csv")
        cols = ["PatientID", "Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"]
        df_clean = preprocess_data(raw_data[cols])

        sample_size = 100  # Adjust this to evaluate more/fewer patients
        if len(df_clean) > sample_size:
            print(f"\nSampling {sample_size} patients from {len(df_clean)} total...")
            df_clean = df_clean.sample(n=sample_size, random_state=42)

        print(f"Generating Schnider parameters for {len(df_clean)} patients...")
        df_sim = generate_schnider_dataset(df_clean)

        print(f"\nRunning evaluation simulation (120 min per patient)...")
        results_df = evaluator.evaluate(df_sim)

        if len(results_df) > 0:
            print("\n" + "=" * 60)
            print("EVALUATION RESULTS")
            print("=" * 60)
            print(f"\nEvaluated {len(results_df)} patients successfully\n")
            print(results_df[['MDPE', 'MDAPE', 'Wobble', 'Controlled (%)']].describe())

            # Best and worst performing patients
            best_idx = results_df['MDAPE'].idxmin()
            worst_idx = results_df['MDAPE'].idxmax()

            print(f"\n--- Best Controlled Patient (ID: {results_df.loc[best_idx, 'PatientID']}) ---")
            print(f"  MDPE: {results_df.loc[best_idx, 'MDPE']:.2f}%")
            print(f"  MDAPE: {results_df.loc[best_idx, 'MDAPE']:.2f}%")
            print(f"  Wobble: {results_df.loc[best_idx, 'Wobble']:.2f}%")
            print(f"  Controlled (%): {results_df.loc[best_idx, 'Controlled (%)']:.2f}%")

            print(f"\n--- Worst Controlled Patient (ID: {results_df.loc[worst_idx, 'PatientID']}) ---")
            print(f"  MDPE: {results_df.loc[worst_idx, 'MDPE']:.2f}%")
            print(f"  MDAPE: {results_df.loc[worst_idx, 'MDAPE']:.2f}%")
            print(f"  Wobble: {results_df.loc[worst_idx, 'Wobble']:.2f}%")
            print(f"  Controlled (%): {results_df.loc[worst_idx, 'Controlled (%)']:.2f}%")
        else:
            print("No patients were successfully evaluated.")

    except Exception as e:
        import traceback
        print(f"ERROR: {e}")
        traceback.print_exc()

